<a href="https://colab.research.google.com/github/murali-github/msai-aih/blob/main/2_Queries_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [2]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated!")

Authenticated!


In [3]:
from google.cloud import bigquery

project_id = 'mc-ut-msai-aih-1'
client = bigquery.Client(project=project_id)
print("BigQuery client connected!")

BigQuery client connected!


In [31]:
# 1. Find patients with multiple hospital admissions

patients_with_multiple_adm_query = """
SELECT
    subject_id,
    COUNT(hadm_id) AS num_admissions
FROM `physionet-data.mimiciii_clinical.admissions`
GROUP BY subject_id
HAVING COUNT(hadm_id) > 1
ORDER BY num_admissions DESC
LIMIT 20
"""

df1 = client.query(patients_with_multiple_adm_query).to_dataframe()

df1_display = df1.rename(columns={
    'subject_id':     'Patient ID',
    'num_admissions': 'Admissions'
})

print(df1_display.to_string(index=False))

 Patient ID  Admissions
13033       42         
  109       34         
11861       34         
 5060       31         
20643       24         
19213       23         
 7809       22         
 5727       21         
23657       20         
11318       19         
23707       17         
73713       17         
19059       17         
 7275       16         
 4787       16         
41976       15         
10635       14         
 3100       14         
25225       14         
19620       14         


In [19]:
# 2. Distribution of days between admissions across all repeat patients

days_between_admissions_query = """
WITH gaps AS (
    SELECT
        subject_id,
        hadm_id,
        DATE_DIFF(
            DATE(admittime),
            DATE(LAG(dischtime) OVER (PARTITION BY subject_id ORDER BY admittime)),
            DAY
        ) AS days_since_last_discharge
    FROM `physionet-data.mimiciii_clinical.admissions`
),
repeat_patients AS (
    SELECT subject_id
    FROM `physionet-data.mimiciii_clinical.admissions`
    GROUP BY subject_id
    HAVING COUNT(hadm_id) > 1
)
SELECT
    CASE
        WHEN days_since_last_discharge <= 7   THEN '1 - Within 7 days'
        WHEN days_since_last_discharge <= 30  THEN '2 - 8 to 30 days'
        WHEN days_since_last_discharge <= 90  THEN '3 - 31 to 90 days'
        WHEN days_since_last_discharge <= 180 THEN '4 - 91 to 180 days'
        WHEN days_since_last_discharge <= 365 THEN '5 - 181 to 365 days'
        ELSE                                       '6 - Over 365 days'
    END AS gap_bucket,
    COUNT(*) AS num_admissions,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct_of_readmissions
FROM gaps g
JOIN repeat_patients r ON g.subject_id = r.subject_id
WHERE days_since_last_discharge IS NOT NULL
  AND days_since_last_discharge >= 0
GROUP BY gap_bucket
ORDER BY gap_bucket
"""

df2 = client.query(days_between_admissions_query).to_dataframe()

# Clean display for screenshot
df2_display = df2.copy()

# Remove sort prefix from bucket labels
df2_display['gap_bucket'] = df2_display['gap_bucket'].str.replace(r'^\d+ - ', '', regex=True)

# Rename columns
df2_display = df2_display.rename(columns={
    'gap_bucket':           'Time Since Last Discharge',
    'num_admissions':       'Admissions',
    'pct_of_readmissions':  '% of Readmissions'
})

print(df2_display.to_string(index=False))

            gap_bucket  num_admissions  pct_of_readmissions
0    1 - Within 7 days            1424                 11.4
1     2 - 8 to 30 days            1995                 16.0
2    3 - 31 to 90 days            2197                 17.6
3   4 - 91 to 180 days            1468                 11.8
4  5 - 181 to 365 days            1535                 12.3
5    6 - Over 365 days            3835                 30.8


In [6]:
# 3. 30-day readmission rate by admission type

readmit_rate_by_type_query = """
WITH gaps AS (
    SELECT
        subject_id,
        hadm_id,
        admission_type,
        DATE_DIFF(
            DATE(admittime),
            DATE(LAG(dischtime) OVER (PARTITION BY subject_id ORDER BY admittime)),
            DAY
        ) AS days_gap
    FROM `physionet-data.mimiciii_clinical.admissions`
)
SELECT
    admission_type,
    COUNT(*) AS total_admissions,
    SUM(CASE WHEN days_gap <= 30 THEN 1 ELSE 0 END) AS readmits_30day,
    ROUND(SUM(CASE WHEN days_gap <= 30 THEN 1 ELSE 0 END) / COUNT(*) * 100, 1) AS readmit_rate_pct
FROM gaps
GROUP BY admission_type
ORDER BY readmit_rate_pct DESC
"""

df3 = client.query(readmit_rate_by_type_query).to_dataframe()

# Clean display for screenshot
df3_display = df3.rename(columns={
    'admission_type':    'Admission Type',
    'total_admissions':  'Total Admissions',
    'readmits_30day':    '30-Day Readmits',
    'readmit_rate_pct':  'Readmit Rate %'
})

print(df3_display.to_string(index=False))

  admission_type  total_admissions  readmits_30day  readmit_rate_pct
0      EMERGENCY             42071            3149               7.5
1         URGENT              1336              73               5.5
2       ELECTIVE              7706             167               2.2
3        NEWBORN              7863              32               0.4


In [7]:
# 4. Top diagnoses driving 30-day readmissions

top_diagnoses_readmission_query = """
WITH gaps AS (
    SELECT
        subject_id,
        hadm_id,
        DATE_DIFF(
            DATE(admittime),
            DATE(LAG(dischtime) OVER (PARTITION BY subject_id ORDER BY admittime)),
            DAY
        ) AS days_gap
    FROM `physionet-data.mimiciii_clinical.admissions`
)
SELECT
    d.short_title,
    COUNT(*) AS readmit_count
FROM gaps g
JOIN `physionet-data.mimiciii_clinical.diagnoses_icd` dx
    ON g.hadm_id = dx.hadm_id AND dx.seq_num = 1
JOIN `physionet-data.mimiciii_clinical.d_icd_diagnoses` d
    ON dx.icd9_code = d.icd9_code
WHERE g.days_gap <= 30
GROUP BY d.short_title
ORDER BY readmit_count DESC
LIMIT 10
"""

df4 = client.query(top_diagnoses_readmission_query).to_dataframe()

# Clean display for screenshot
df4_display = df4.rename(columns={
    'short_title':    'Primary Diagnosis',
    'readmit_count':  'Readmissions'
})

print(df4_display.to_string(index=False))

                short_title  readmit_count
0            Septicemia NOS            210
1  Acute respiratry failure            130
2    Other postop infection            109
3  Fetal/neonatal jaund NOS            105
4    Food/vomit pneumonitis             87
5                   CHF NOS             61
6   Gastrointest hemorr NOS             57
7  Acute kidney failure NOS             55
8   Pneumonia, organism NOS             52
9  Acute & chronc resp fail             51


In [8]:
# 5. Who gets readmitted - gender breakdown of readmitted vs single-admission patients

readmit_demographics_query = """
SELECT
    p.gender,
    CASE
        WHEN a.num_adm > 1 THEN 'Readmitted'
        WHEN a.num_adm = 1 THEN 'Single Admission'
        ELSE 'No Admission Record'
    END AS readmit_status,
    COUNT(*) AS patient_count
FROM `physionet-data.mimiciii_clinical.patients` p
LEFT JOIN (
    SELECT subject_id, COUNT(hadm_id) AS num_adm
    FROM `physionet-data.mimiciii_clinical.admissions`
    GROUP BY subject_id
) a
ON p.subject_id = a.subject_id
GROUP BY p.gender, readmit_status
ORDER BY p.gender, readmit_status
"""

df5 = client.query(readmit_demographics_query).to_dataframe()

# Clean display for screenshot
df5_display = df5.copy()

# Add readmission rate column calculated in pandas
total_by_gender = df5_display.groupby('gender')['patient_count'].transform('sum')
df5_display['readmit_rate_pct'] = (df5_display['patient_count'] / total_by_gender * 100).round(1)

# Rename columns
df5_display = df5_display.rename(columns={
    'gender':          'Gender',
    'readmit_status':  'Status',
    'patient_count':   'Patients',
    'readmit_rate_pct': '% of Gender'
})

print(df5_display.to_string(index=False))

  gender    readmit_status  patient_count
0      F        Readmitted           3347
1      F  Single Admission          17052
2      M        Readmitted           4190
3      M  Single Admission          21931


In [20]:
# 6. How common is heavy lab repetition within a single admission?

repeated_lab_tests_query = """
WITH test_counts AS (
    SELECT
        l.subject_id,
        l.hadm_id,
        TRIM(UPPER(d.label)) AS test_name,
        COUNT(*) AS times_tested
    FROM `physionet-data.mimiciii_clinical.labevents` l
    JOIN `physionet-data.mimiciii_clinical.d_labitems` d
        ON l.itemid = d.itemid
    WHERE l.hadm_id IS NOT NULL
    GROUP BY l.subject_id, l.hadm_id, test_name
)
SELECT
    CASE
        WHEN times_tested >= 100 THEN '1 - 100+ draws'
        WHEN times_tested >= 50  THEN '2 - 50 to 99 draws'
        WHEN times_tested >= 20  THEN '3 - 20 to 49 draws'
        WHEN times_tested >= 10  THEN '4 - 10 to 19 draws'
        ELSE                          '5 - 6 to 9 draws'
    END AS repetition_bucket,
    COUNT(*) AS num_patient_test_pairs,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct_of_total
FROM test_counts
WHERE times_tested > 5
GROUP BY repetition_bucket
ORDER BY repetition_bucket
"""

df6 = client.query(repeated_lab_tests_query).to_dataframe()

print(df6)

    repetition_bucket  num_patient_test_pairs  pct_of_total
0      1 - 100+ draws                    6408           0.6
1  2 - 50 to 99 draws                   34428           3.2
2  3 - 20 to 49 draws                  193610          18.1
3  4 - 10 to 19 draws                  391261          36.6
4    5 - 6 to 9 draws                  442739          41.4


In [25]:
# 7. Average lab draws per admission by readmission frequency group

lab_vs_readmit_frequency_query = """
WITH admission_counts AS (
    SELECT
        subject_id,
        COUNT(hadm_id) AS num_admissions
    FROM `physionet-data.mimiciii_clinical.admissions`
    GROUP BY subject_id
),
lab_counts AS (
    SELECT
        subject_id,
        COUNT(*) AS total_lab_draws
    FROM `physionet-data.mimiciii_clinical.labevents`
    WHERE hadm_id IS NOT NULL
    GROUP BY subject_id
)
SELECT
    CASE
        WHEN a.num_admissions = 1   THEN '1 - Single Admission'
        WHEN a.num_admissions <= 3  THEN '2 - 2 to 3 Admissions'
        WHEN a.num_admissions <= 6  THEN '3 - 4 to 6 Admissions'
        WHEN a.num_admissions <= 10 THEN '4 - 7 to 10 Admissions'
        ELSE                             '5 - 10+ Admissions'
    END AS readmission_group,
    COUNT(*) AS num_patients,
    ROUND(AVG(l.total_lab_draws), 0) AS avg_total_lab_draws,
    ROUND(AVG(l.total_lab_draws / a.num_admissions), 0) AS avg_labs_per_admission
FROM admission_counts a
JOIN lab_counts l ON a.subject_id = l.subject_id
GROUP BY readmission_group
ORDER BY readmission_group
"""

df7 = client.query(lab_vs_readmit_frequency_query).to_dataframe()

# Clean display for screenshot
df7_display = df7.copy()

# Remove sort prefix from group labels
df7_display['readmission_group'] = df7_display['readmission_group'].str.replace(r'^\d+ - ', '', regex=True)

# Shorten column names
df7_display = df7_display.rename(columns={
    'readmission_group':      'Admission Group',
    'num_patients':           'Patients',
    'avg_total_lab_draws':    'Avg Total Labs',
    'avg_labs_per_admission': 'Avg Labs/Admission'
})

print(df7_display.to_string(index=False))

Admission Group     Patients  Avg Total Labs  Avg Labs/Admission
  Single Admission 38665      348.0          348.0              
 2 to 3 Admissions  6501      966.0          437.0              
 4 to 6 Admissions   867     2150.0          478.0              
7 to 10 Admissions   122     3289.0          410.0              
    10+ Admissions    46     5049.0          347.0              


In [24]:
# 8. Clean display for screenshot

df8_display = df8.copy()

# Remove the sort prefix from bucket labels
df8_display['abnormal_bucket'] = df8_display['abnormal_bucket'].str.replace(r'^\d+ - ', '', regex=True)

# Shorten column names
df8_display = df8_display.rename(columns={
    'abnormal_bucket': 'Abnormal Range',
    'patient_count': 'Patients',
    'pct_of_patients': '% of Total',
    'avg_draws_per_patient': 'Avg Draws',
    'avg_abnormal_pct': 'Avg Abnormal %'
})

pd.set_option('display.max_columns', None)      # show all columns
pd.set_option('display.width', 1000)            # wide display, no line wrapping
pd.set_option('display.max_colwidth', 30)       # truncate long text cells
pd.set_option('display.colheader_justify', 'left')  # left align headers

print(df8_display.to_string(index=False))

Abnormal Range   Patients  % of Total  Avg Draws  Avg Abnormal %
  100% Abnormal   667      2.7        14.4       100.0          
75-99% Abnormal  9532     37.9        30.3        83.8          
50-74% Abnormal 10558     42.0        34.8        63.4          
25-49% Abnormal  3678     14.6        29.0        38.8          
 1-24% Abnormal   689      2.7        21.2        17.1          
    0% Abnormal    35      0.1        12.2         0.0          


In [30]:
# 9. Repeat behavior - same vs different diagnosis on consecutive admissions + population summary

repeat_diagnosis_query = """
WITH primary_dx AS (
    SELECT
        a.subject_id,
        a.hadm_id,
        a.admittime,
        dx.icd9_code,
        d.short_title
    FROM `physionet-data.mimiciii_clinical.admissions` a
    JOIN `physionet-data.mimiciii_clinical.diagnoses_icd` dx
        ON a.hadm_id = dx.hadm_id AND dx.seq_num = 1
    JOIN `physionet-data.mimiciii_clinical.d_icd_diagnoses` d
        ON dx.icd9_code = d.icd9_code
),
with_lag AS (
    SELECT
        subject_id,
        hadm_id,
        icd9_code,
        short_title,
        LAG(icd9_code) OVER (PARTITION BY subject_id ORDER BY admittime) AS prev_icd9_code
    FROM primary_dx
)
SELECT
    CASE
        WHEN icd9_code = prev_icd9_code THEN 'Same Diagnosis'
        ELSE 'Different Diagnosis'
    END AS recurrence_flag,
    COUNT(*) AS num_admissions,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct_of_admissions
FROM with_lag
WHERE prev_icd9_code IS NOT NULL
GROUP BY recurrence_flag
ORDER BY recurrence_flag DESC
"""

df9 = client.query(repeat_diagnosis_query).to_dataframe()

# Clean display for screenshot
df9_display = df9.rename(columns={
    'recurrence_flag':    'Recurrence',
    'num_admissions':     'Admissions',
    'pct_of_admissions':  '% of Total'
})

print(df9_display.to_string(index=False))

Recurrence           Admissions  % of Total
     Same Diagnosis  1566       13.0       
Different Diagnosis 10517       87.0       


In [18]:
# 10. High-utilizer overlap - patients who are both frequently readmitted AND heavily lab-tested

high_utilizer_overlap_query = """
WITH frequent_readmitters AS (
    SELECT
        subject_id,
        COUNT(hadm_id) AS num_admissions
    FROM `physionet-data.mimiciii_clinical.admissions`
    GROUP BY subject_id
    HAVING COUNT(hadm_id) > 2
),
frequent_testers AS (
    SELECT
        subject_id,
        COUNT(*) AS num_lab_draws
    FROM `physionet-data.mimiciii_clinical.labevents`
    WHERE hadm_id IS NOT NULL
    GROUP BY subject_id
    HAVING COUNT(*) > 200
)
SELECT
    r.subject_id,
    r.num_admissions,
    t.num_lab_draws,
    RANK() OVER (ORDER BY r.num_admissions DESC, t.num_lab_draws DESC) AS utilizer_rank
FROM frequent_readmitters r
JOIN frequent_testers t ON r.subject_id = t.subject_id
ORDER BY utilizer_rank
LIMIT 20
"""

df10 = client.query(high_utilizer_overlap_query).to_dataframe()

print(df10)

    subject_id  num_admissions  num_lab_draws  utilizer_rank
0        13033              42           7872              1
1          109              34           7483              2
2        11861              34           7208              3
3         5060              31           3466              4
4        20643              24           4375              5
5        19213              23           9653              6
6         7809              22           6373              7
7         5727              21           4810              8
8        23657              20           3362              9
9        11318              19          10080             10
10       73713              17           5119             11
11       19059              17           3697             12
12       23707              17           2567             13
13        4787              16           4458             14
14        7275              16           2292             15
15       41976          